# 7.16 Transformers

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook builds from attention into the transformer architecture and keeps the focus on the practical components used in modern sequence models.


## 7.16.1 Transformer block anatomy

### Why It Matters
A transformer block combines self-attention, feedforward layers, normalization, and residual connections. PyTorch exposes this structure directly.


In [ ]:
import torch
from torch import nn

block = nn.TransformerEncoderLayer(d_model=16, nhead=4, dim_feedforward=32, batch_first=True)
x = torch.randn(3, 5, 16)
out = block(x)
{"input_shape": list(x.shape), "output_shape": list(out.shape)}


### Interpretation
The block preserves sequence length and embedding width while transforming the representation.


## 7.16.2 Multi-head attention intuition

### Why It Matters
Multiple heads let the model attend to different relationships in parallel. The easiest way to see that is through the tensor shapes.


In [ ]:
import torch
from torch import nn

mha = nn.MultiheadAttention(embed_dim=12, num_heads=3, batch_first=True)
x = torch.randn(2, 4, 12)
out, weights = mha(x, x, x, need_weights=True)
{"output_shape": list(out.shape), "attention_weight_shape": list(weights.shape)}


### Interpretation
The heads are internal, but the effect is that several attention patterns can be learned at once.


## 7.16.3 Positional encoding

### Why It Matters
Self-attention alone does not know token order, so transformers add positional information. A sinusoidal encoding makes the idea inspectable without training.


In [ ]:
import math
import torch

seq_len, d_model = 5, 6
positions = torch.arange(seq_len).unsqueeze(1)
div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
pe = torch.zeros(seq_len, d_model)
pe[:, 0::2] = torch.sin(positions * div_term)
pe[:, 1::2] = torch.cos(positions * div_term)
pe.round(decimals=3)


### Interpretation
The values encode position in a way the model can combine with token embeddings.


## 7.16.4 Encoder versus decoder framing

### Why It Matters
Encoders map an input sequence to contextual representations; decoders generate outputs step by step, often with masking.


In [ ]:
import torch
from torch import nn

encoder = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(d_model=8, nhead=2, batch_first=True),
    num_layers=1,
)
decoder = nn.TransformerDecoder(
    nn.TransformerDecoderLayer(d_model=8, nhead=2, batch_first=True),
    num_layers=1,
)

source = torch.randn(2, 5, 8)
target = torch.randn(2, 4, 8)
memory = encoder(source)
causal_mask = nn.Transformer.generate_square_subsequent_mask(target.size(1))
decoded = decoder(target, memory, tgt_mask=causal_mask)

{
    "encoder_memory_shape": list(memory.shape),
    "decoder_output_shape": list(decoded.shape),
    "causal_mask_shape": list(causal_mask.shape),
}


### Interpretation
The distinction matters because it changes the flow of information and the training objective.


## 7.16.5 Masking

### Why It Matters
Autoregressive decoders must not look ahead at future tokens. A causal mask enforces that constraint inside attention.


In [ ]:
import torch
from torch import nn

seq_len = 4
mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1)
mha = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
x = torch.randn(1, seq_len, 8)
out, weights = mha(x, x, x, attn_mask=mask, need_weights=True)
{"causal_mask": mask.tolist(), "attention_weights_shape": list(weights.shape)}


### Interpretation
The mask prevents each position from using information from tokens that come later in the sequence.


## 7.16.6 A compact transformer classifier

### Why It Matters
This final subsection builds a tiny transformer encoder for sequence classification on a toy token dataset.


In [ ]:
import torch
from torch import nn

torch.manual_seed(40)
X = torch.randint(0, 20, (120, 6))
y = ((X[:, 0] + X[:, 1]) > 18).long()

class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=32, batch_first=True),
            num_layers=1,
        )
        self.head = nn.Linear(d_model, num_classes)
    def forward(self, x):
        embedded = self.embedding(x)
        encoded = self.encoder(embedded)
        pooled = encoded.mean(dim=1)
        return self.head(pooled)

model = TinyTransformerClassifier(20, 12, 3, 2)
opt = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.CrossEntropyLoss()
for _ in range(40):
    logits = model(X)
    loss = loss_fn(logits, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
with torch.no_grad():
    preds = model(X).argmax(dim=1)
{"training_accuracy": round(float((preds == y).float().mean().item()), 3), "final_loss": round(float(loss.item()), 4)}


### Interpretation
The model is tiny, but it already uses the same building blocks as much larger transformer systems.
